In [ ]:
import os
import subprocess

# Find the root of the current git repository (usually your uv project root)
root = 'f:\\Projects\\GeneIndex'
os.chdir(root)

print(f"Current working directory: {os.getcwd()}")

In [ ]:
from src.u1_downloader.labs_downloader import get_all_rid, get_data_from_rid, extract_from_extras
import time
import pandas as pd
from src.u1_downloader.images_downloader import download_volume, get_filenames
import numpy as np
from bs4 import BeautifulSoup
import requests
from bs4 import BeautifulSoup
import requests
from functools import cache

In [ ]:
all_rid = get_all_rid()

data = []
for rid in all_rid[all_rid.type=='birth'].sample(5).rid:
    data.extend(get_data_from_rid(rid))

df_labs = pd.DataFrame(data,columns=['born_year','act_num','name','surname','father_name','mother_name','mother_surname','parish','location','extras'])
df_labs[['extras_i','extras_z','extras_a','source_link']] = extract_from_extras(df_labs.extras)

In [ ]:
from src.u1_downloader.labs_tools import get_img_id_from_labels

In [ ]:
df_labs[df_labs.source_link != ''].iloc[5].source_link

In [ ]:
def _filter_mask_easy(df: pd.DataFrame):
    return ((df.act_num != '') * (df.name != '') * (df.surname != '')*df.source_link.str.startswith('https://metryki.genealodzy.pl/'))


@cache
def _get_gid_from_row(ar,zsid):
    soup = BeautifulSoup(requests.get(f'https://skanoteka.genealodzy.pl/metryki.php?op=kt&ar={ar}&zs={zsid}').text)
    # print('Out: ', soup.find('a', string="powrót do zespołu").get('href')[2::])
    return soup.find('a', string="powrót do zespołu").get('href')[2::]


def get_img_id_from_labels(df):
    df_links = df[_filter_mask_easy(df)].source_link.str[len(' https://metryki.genealodzy.pl/')-1::].reset_index(drop=True)

    df_down = pd.DataFrame(columns=['gid','zsid','sgid','vol'])
    df_down["gid"] = df_links.str.extract(r'(?:[?&]id=|\bid)([a-zA-Z0-9]+)')
    df_down["zsid"] = df_links.str.extract(r'(?:[?&]zs=|\bzs)([a-zA-Z0-9]+)')
    df_down["sgid"] = df_links.str.extract(r'(?:[?&]sy=|-sy)([a-zA-Z0-9]+)')
    df_down["vol"] = df_links.str.extract(r'(?:[?&]kt=|-kt)([a-zA-Z0-9]+)')

    df_down.loc[df_down.gid.isna(), 'gid'] = df_down.loc[df_down.gid.isna(), 'zsid'].apply(_get_gid_from_zsid)
    df_down = df_down.drop('zsid',axis=1)

    return df_down


In [ ]:
df_links = df_labs[_filter_mask_easy(df_labs)].source_link.str[len(' https://metryki.genealodzy.pl/')-1::].reset_index(drop=True)

df_down = pd.DataFrame(columns=['gid','zsid', 'ar','sgid','vol'])
df_down["gid"] = df_links.str.extract(r'(?:[?&]id=|\bid)([a-zA-Z0-9]+)')
df_down["zsid"] = df_links.str.extract(r'(?:[?&]zs=|\bzs)([a-zA-Z0-9]+)')
df_down["ar"] = df_links.str.extract(r'(?:[?&]ar=|\bar)([a-zA-Z0-9]+)')
df_down["sgid"] = df_links.str.extract(r'(?:[?&]sy=|-sy)([a-zA-Z0-9]+)')
df_down["vol"] = df_links.str.extract(r'(?:[?&]kt=|-kt)([a-zA-Z0-9]+)')

In [ ]:
df_down.ar.describe()

In [ ]:
( df_down.ar.isna() * df_down.gid.isna() ).sum()

In [ ]:
df_down.loc[df_down.gid.isna(), 'gid'] = df_down.loc[df_down.gid.isna(), :].apply(lambda x: _get_gid_from_row(x.ar,x.zsid),axis=1)
df_down = df_down.drop(['zsid','ar'],axis=1)

In [ ]:
df_down

In [ ]:
for row in df_down.iloc:
    download_subgroup('datasets/full_scans_from_labs',row.gid, row.sgid, row.vol)